# ECG Demo — Overlay Diagnostic Notebook
Run each cell in order to verify the overlay loads correctly before starting the server.

In [ ]:
# Cell 1 — Load overlay and list available IP cores
from pynq import Overlay

overlay = Overlay('ecg_demo.bit')
axi = overlay.axi_ecg_ctrl

print('Overlay loaded successfully.')
print('IP cores in overlay:')
for name, ip in overlay.ip_dict.items():
    print(f'  {name}: {ip["type"]}')

In [ ]:
# Cell 2 — Read all 17 registers and print as a table
REGISTERS = [
    (0x00, 'BPM_CH_A',         0xFF,   'R/W'),
    (0x04, 'RR_FLUCT',         0xFF,   'R/W'),
    (0x08, 'AMP_FLUCT',        0xFF,   'R/W'),
    (0x0C, 'BPM_CH_B',         0xFF,   'R/W'),
    (0x10, 'BPM_CH_C',         0xFF,   'R/W'),
    (0x14, 'BPM_CH_D',         0xFF,   'R/W'),
    (0x18, 'BPM_CH_E',         0xFF,   'R/W'),
    (0x1C, 'BPM_CH_F',         0xFF,   'R/W'),
    (0x20, 'BPM_CH_G',         0xFF,   'R/W'),
    (0x24, 'BPM_CH_H',         0xFF,   'R/W'),
    (0x28, 'ECG_RAW',          0xFFF,  'R'),
    (0x2C, 'ECG_FILTERED',     0xFFF,  'R'),
    (0x30, 'BPM_OUT',          0xFF,   'R'),
    (0x34, 'RPEAK_COUNT',      0xFFFF, 'R'),
    (0x38, 'DETECT_THRESHOLD', 0xFFF,  'R/W'),
    (0x3C, 'STATUS',           0x03,   'R'),
    (0x40, 'ECG_DAC',          0xFFF,  'R'),
]

print(f'{'Offset':<8} {'Name':<20} {'Raw (hex)':<12} {'Masked':<10} {'R/W'}')
print('-' * 60)
for offset, name, mask, rw in REGISTERS:
    raw = axi.read(offset)
    masked = raw & mask
    print(f'0x{offset:02X}     {name:<20} 0x{raw:08X}   {masked:<10} {rw}')

In [ ]:
# Cell 3 — Write BPM_CH_A = 90, read back and verify
BPM_CH_A_OFFSET = 0x00
TARGET_BPM = 90

axi.write(BPM_CH_A_OFFSET, TARGET_BPM)
readback = axi.read(BPM_CH_A_OFFSET) & 0xFF

assert readback == TARGET_BPM, f'Readback mismatch: expected {TARGET_BPM}, got {readback}'
print(f'BPM_CH_A write/readback OK: {readback} BPM')

In [ ]:
# Cell 4 — Poll ECG_RAW 10 times at 100 ms intervals
import time

ECG_RAW_OFFSET = 0x28
SAMPLES = 10
INTERVAL_S = 0.1

print(f'Polling ECG_RAW {SAMPLES} times at {int(INTERVAL_S * 1000)} ms intervals:')
for i in range(SAMPLES):
    val = axi.read(ECG_RAW_OFFSET) & 0xFFF
    print(f'  [{i:2d}]  ECG_RAW = 0x{val:03X}  ({val})')
    time.sleep(INTERVAL_S)

In [ ]:
# Cell 5 — Check STATUS register; assert signal_present bit
STATUS_OFFSET = 0x3C

status_raw = axi.read(STATUS_OFFSET) & 0x03
signal_present = bool(status_raw & 0x01)
lead_off       = bool(status_raw & 0x02)

print(f'STATUS register: 0x{status_raw:02X}')
print(f'  signal_present : {signal_present}')
print(f'  lead_off       : {lead_off}')

assert signal_present, (
    'signal_present is NOT set — check that ECG_RAW > 0x010 and the PL is running.'
)
print('PASS: signal_present asserted.')